# Giving Agents Long-Term Memory

The context window is an agent's working memory — every token the model can see during a single generation pass. It is fast, immediately available, and zero-latency. It is also finite and volatile: when the session ends, everything in it disappears. A 200,000-token context window sounds generous until you realize it can be exhausted by a handful of large file reads, and even within a session, information buried in the middle of a long history is attended to less reliably than information at the edges.

Agents that can operate across sessions, recall user preferences, consult stable project facts, and track pending work are qualitatively more useful than agents that start fresh every time. This notebook covers the five types of memory an agent can leverage, how the CDA library implements each one, and how to wire them together into a coherent memory strategy. We ground each type in a working code example so the abstract taxonomy connects directly to the library's APIs.

Two papers shape this landscape: Generative Agents [@genagents] introduced a **memory stream** architecture where observations, reflections, and plans are stored and retrieved by recency, importance, and relevance. MemGPT [@memgpt] formalized an OS-inspired memory hierarchy, distinguishing in-context ("main memory") from out-of-context ("external storage") and giving the agent explicit control over what moves between them. We take both seriously and map their concepts onto the CDA library.

## A Taxonomy of Agent Memory

Five types of memory are useful to distinguish when designing agents:

| Type | Analogy | Lifetime | CDA Implementation |
|------|---------|----------|-------------------|
| Working | RAM | One context window | `Session.messages` |
| Episodic | Diary | Across sessions | `Session.save()` / `Session.load()` |
| Semantic | Fact database | Persistent | `MemoryTool` (key-value store) |
| Procedural | Muscle memory | Long-lived | `Config.developer_instructions` |
| Prospective | To-do list | Until completed | `TodoTool` |

: {tbl-colwidths="[18,18,24,40]"}

<br>

**Working memory** is the context window itself — the full set of tokens the model can attend to during a single generation. The CDA library manages it through `Session.messages`. Everything in the current session lives here: system prompt, user turns, assistant replies, tool results. NB04 covered strategies for keeping it from overflowing.

**Episodic memory** is the record of what happened in prior sessions. For a human, it is autobiographical narrative: "last Tuesday I diagnosed a segfault in auth.py." For an agent, it is the serialized conversation history from a previous run. `Session.save()` writes this to disk; `Session.load()` restores it into a new session so the agent can continue where it left off.

**Semantic memory** is long-lived factual knowledge: project-specific constants, user preferences, stable configurations. Unlike episodic memory (which is a narrative), semantic memory is a lookup table. The CDA library implements this as `MemoryTool`, a persistent key-value store at `~/.cda/memory.json`.

**Procedural memory** encodes how-to knowledge: workflows, constraints, style rules. Agents don't learn new procedures at runtime the way humans do, but the system prompt injects procedural knowledge via `Config.developer_instructions`. Instructions like "run `ruff check` before treating any code change as complete" are effectively procedural memories activated on every turn.

**Prospective memory** is the intention to do something in the future — the mental note you make when you can't act immediately. The CDA library models this as `TodoTool`, a persistent checklist the agent can write to and consult across turns.

:::{.callout-note}
This notebook focuses on the three most important types for practical agents: working memory (the context window), episodic memory (session persistence), and semantic memory (the `MemoryTool`). Procedural and prospective memory are covered briefly in the final section.

:::

## Working Memory — The Context Window

The model's working memory is the context window — the complete token sequence the model attends to at generation time. It feels uniform, but it isn't. Liu et al. (2023) demonstrated the **"lost in the middle" phenomenon**: models attend most strongly to tokens near the beginning and end of the context. Performance on retrieval and reasoning tasks degrades for information buried in the middle of a long prompt.

For agents, this has a concrete design implication: critical facts — user preferences, the current task goal, project-specific constraints — should appear near the start of the context (in the system prompt) or near the end (in recent turns), not buried in old tool outputs. The system prompt structure from NB03 and NB04 was designed with this in mind: identity and instructions appear first, giving them the strongest positional attention.

**Experiment.** We demonstrate the position effect directly. A long filler context is constructed with a target fact placed at the start, middle, or end. We ask the model to recall it and observe whether position affects reliability.

**Setup.** Imports and client initialization:

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
MODEL = "anthropic/claude-sonnet-4"

Importing from the CDA library:

In [ ]:
from notebooks.agent.config import Config
from notebooks.agent.session import Session, SESSIONS_DIR
from notebooks.agent.agent import Agent
from notebooks.agent.events import AgentEventType
from notebooks.agent.tools.builtin.memory import MemoryTool
from notebooks.agent.tools.base import ToolInvocation

The position experiment: we embed the target fact at the start, middle, or end of a long filler passage and ask the model to recall it:

In [ ]:
async def position_test(position: str, context_size: int = 50) -> str:
    """Test recall of a fact at different positions in a long context."""
    filler = "The sky is blue. The grass is green. Numbers are abstract objects. " * context_size
    target_fact = "The secret code is ALPHA-7."

    if position == "start":
        context = target_fact + " " + filler
    elif position == "end":
        context = filler + " " + target_fact
    else:  # middle
        half = len(filler) // 2
        context = filler[:half] + " " + target_fact + " " + filler[half:]

    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Context: {context}\n\nWhat is the secret code mentioned in the context?"},
        ],
        max_tokens=50,
    )
    return response.choices[0].message.content.strip()


for pos in ["start", "middle", "end"]:
    answer = await position_test(pos)
    print(f"{pos:8s}: {answer}")

:::{.callout-note}
For agents, the practical implication is to keep critical facts near the start of the context — in the system prompt or early in the conversation — where they receive the strongest positional attention. `Config.developer_instructions` and `Config.user_instructions` are injected into the system prompt precisely for this reason. Long tool results that accumulate in the middle of the history are the most vulnerable to being effectively "forgotten" by the model.

:::

## Episodic Memory — Session Persistence

Episodic memory is the record of what happened. In a human, it is autobiographical: "last Tuesday I fixed a segfault in `auth.py`." For an agent, it is the saved conversation history from a prior session — the full sequence of user turns, assistant reasoning, tool calls, and tool results that constitutes a completed work session.

`Session.save(name)` serializes the session to `~/.cda/sessions/<name>.json`. `Session.load(name, config)` restores it into a new `Session` so the agent can resume from where it left off, with the full prior context available for reference. The serialized format stores (1) the complete message history, (2) the turn count, and (3) accumulated token usage. Config is intentionally not stored — credentials and environment-specific settings belong in the environment, not in saved files.

We walk through the full save/load cycle with a synthetic conversation:

**Setup.** We build a session manually, adding a short JWT refactoring conversation as history:

In [ ]:
config = Config()
session = Session(config)

session.add_user_message("Refactor the auth module to use JWT.")
session.add_assistant_message("I'll start by reading auth.py.")
session.add_user_message("Also add refresh token support.")
session.add_assistant_message("Understood. I've added refresh token support.")
session.turn_count = 2

path = session.save("demo-session")
print(f"Saved to:            {path}")
print(f"Messages in session: {len(session.messages)}")

Loading the session into a fresh `Session` object restores the full message history and metadata:

In [ ]:
loaded = Session.load("demo-session", config)
print(f"Loaded {len(loaded.messages)} messages")
print(f"Turn count: {loaded.turn_count}")
print("\nMessage history:")
for msg in loaded.messages:
    role = msg["role"]
    content = str(msg.get("content", ""))[:80].replace("\n", " ")
    print(f"  [{role:10s}] {content}")

Inspecting the raw JSON on disk shows the serialization format:

In [ ]:
raw = json.loads(path.read_text())
print(json.dumps({k: v for k, v in raw.items() if k != "messages"}, indent=2))
print(f"\nmessages: {len(raw['messages'])} items")

:::{.callout-caution}
Config is **not** persisted — model name, temperature, approval policy, and custom instructions must be provided again when loading a session. This is intentional: credentials and environment-specific configuration should never be embedded in saved files. The same saved session can be loaded under different configs: a development config with `approval=AUTO` and a production config with `approval=ON_REQUEST`, for instance.

:::

`Session.list_saved()` enumerates saved sessions in the default directory:

In [ ]:
sessions = Session.list_saved()
print(f"Saved sessions: {sessions}")

# Clean up the demo session
(SESSIONS_DIR / "demo-session.json").unlink(missing_ok=True)
print("Demo session removed.")

## The Sentinel-Manager-Executor Pipeline

Not every message warrants a memory operation. Most user utterances are conversational — requests, acknowledgments, clarifications — that don't contain facts worth storing. Applying expensive memory logic on every turn wastes tokens and can pollute the store with noise.

The **sentinel-manager-executor** (SME) pipeline solves this with a three-stage soft filter:

1. **Sentinel** — a binary gate that reads the user message and returns `TRUE` or `FALSE`: does this contain new information worth remembering? The sentinel uses a fast, low-temperature call to avoid over-filtering.
2. **Manager** — if the sentinel fires, the manager extracts the specific fact and formulates a key-value pair: what to store and under what key.
3. **Executor** — writes the extracted fact to the memory store.

The gate dramatically reduces the number of memory writes in a typical session. Instead of storing every message, only the semantically significant ones (facts, preferences, constraints, decisions) are persisted.

We implement the sentinel and manager as raw LLM calls:

In [ ]:
SENTINEL_SYSTEM = """You are a classifier. Decide whether a user message contains new information worth storing as a persistent memory (facts, preferences, important decisions, project-specific rules).

Return exactly one word: TRUE or FALSE. No other text."""

MANAGER_SYSTEM = """You are a memory manager. Extract the key fact from the user message and return a JSON object:
{"key": "short_descriptive_key", "value": "the fact to remember"}

Return only the JSON object, no other text. Use snake_case for keys."""


async def sentinel(message: str) -> bool:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SENTINEL_SYSTEM},
            {"role": "user", "content": message},
        ],
        max_tokens=5,
        temperature=0.0,
    )
    result = response.choices[0].message.content.strip().upper()
    return result == "TRUE"


async def manager(message: str) -> dict:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": MANAGER_SYSTEM},
            {"role": "user", "content": message},
        ],
        max_tokens=100,
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)

We wire the two stages into a `process_message()` function that routes each user message through the pipeline and writes to an in-memory dict as the store:

In [ ]:
memory: dict[str, str] = {}


async def process_message(message: str) -> dict:
    """Route a user message through the SME pipeline."""
    should_store = await sentinel(message)      # <1>
    if not should_store:
        return {"stored": False, "message": message}

    extracted = await manager(message)          # <2>
    key = extracted.get("key", "unknown")
    value = extracted.get("value", message)
    memory[key] = value                         # <3>
    return {"stored": True, "key": key, "value": value}


test_messages = [
    "Hi, how are you?",
    "My name is Alice.",
    "I prefer Python over JavaScript.",
    "What time is it?",
    "The database runs on port 5432.",
    "Thanks, that's all for now.",
]

for msg in test_messages:
    result = await process_message(msg)
    tag = "\u2713 stored" if result["stored"] else "  skipped"
    if result["stored"]:
        print(f"{tag} | {result['key']}: {result['value']}")
    else:
        print(f"{tag} | {msg!r}")

print(f"\nMemory store ({len(memory)} entries): {memory}")

1. The sentinel is called first with a low-temperature, short-max-tokens setting — a fast, cheap binary decision.
2. Only if the sentinel returns `TRUE` do we pay for the heavier manager call to extract the structured fact.
3. Here we write to a local dict. In a live agent session the executor would call `MemoryTool(action="set", key=..., value=...)` instead.

:::{.callout-note}
The SME pipeline is a soft filter — the sentinel will occasionally miss important facts or flag trivial ones. In production, combine it with explicit `memory(action="set", ...)` calls from the agent itself (NB03 showed the agent can call `MemoryTool` directly). The pipeline is best suited to passively enriching the store during conversation, not replacing deliberate agent memory writes.

:::

## Semantic Memory — The MemoryTool

The `MemoryTool` is the CDA library's built-in persistent key-value store for semantic memory. It persists facts at `~/.cda/memory.json` across agent runs. The agent calls it like any other tool — `memory(action="set", key="db_port", value="5432")` — and the result survives session boundaries. The five supported actions are:

- `set` — store a value under a key (overwrites if key exists)
- `get` — retrieve the value for a key
- `delete` — remove a key-value pair
- `list` — enumerate all stored keys with value previews
- `clear` — delete all entries

The tool's `kind` is `ToolKind.MEMORY`. In `ApprovalPolicy.ON_REQUEST` mode this means the user is prompted before any write — the same gate that protects file writes and shell commands. For fully automated pipelines, `ApprovalPolicy.AUTO` skips the prompt.

We exercise the tool directly via `ToolInvocation`:

In [ ]:
tool = MemoryTool()

result = await tool.execute(ToolInvocation(params={"action": "set", "key": "db_port", "value": "5432"}))
print("set db_port:          ", result.output)

result = await tool.execute(ToolInvocation(params={"action": "set", "key": "preferred_language", "value": "Python 3.13"}))
print("set preferred_language:", result.output)

result = await tool.execute(ToolInvocation(params={"action": "set", "key": "project_name", "value": "ai-notebooks"}))
print("set project_name:     ", result.output)

result = await tool.execute(ToolInvocation(params={"action": "list"}))
print("\nAll keys:\n", result.output)

result = await tool.execute(ToolInvocation(params={"action": "get", "key": "db_port"}))
print("\ndb_port:", result.output)

Cleaning up the demo entries after the demonstration:

In [ ]:
for key in ["db_port", "preferred_language", "project_name"]:
    await tool.execute(ToolInvocation(params={"action": "delete", "key": key}))
print("Demo entries cleaned up.")

:::{.callout-note}
`MemoryTool` is exact-key-match only. If you don't know the key for a fact, use `list` first to enumerate what's stored. For fuzzy recall — "find memories related to the database configuration" — exact key lookup is insufficient. The natural extension is embedding-based retrieval: vectorize each stored value at write time, store the embeddings alongside the keys, and at query time compute cosine similarity between the query embedding and all stored embeddings to surface the most relevant entries. This is the pattern used by long-term memory systems like LangMem and MemGPT. We don't implement it here, but the `MemoryTool`'s `list` action gives you the full store to work with if you want to build that layer on top.

:::

## Procedural & Prospective Memory

The remaining two memory types complete the taxonomy. Both are simpler than episodic and semantic memory — they don't require persistence mechanisms that the agent actively manages at runtime — but they are important for well-behaved, predictable agents.

**Procedural memory** encodes how to do things: workflows, style rules, project constraints. For agents, this lives in the system prompt via `Config.developer_instructions`. Instructions injected here are "always on" — activated on every single turn without the agent needing to look them up.

We construct a config with explicit procedural instructions and inspect the system prompt section they appear in:

In [ ]:
from notebooks.agent.prompts import build_system_prompt

config_procedural = Config(
    developer_instructions=(
        "Always write Python code that is compatible with Python 3.13+.\n"
        "Run `ruff check` before treating any code change as complete.\n"
        "Never modify files in the `migrations/` directory without explicit user approval."
    )
)

prompt = build_system_prompt(config_procedural, [])
lines = prompt.split("\n")
in_section = False
for i, line in enumerate(lines):
    if "# Project Instructions" in line:
        in_section = True
    if in_section:
        print(line)
    if in_section and i > 0 and line.startswith("# ") and "Project Instructions" not in line:
        break

The developer instructions appear verbatim in the system prompt under the `# Project Instructions` heading, placed early in the prompt where positional attention is strongest.

<br>

**Prospective memory** encodes future intentions — tasks the agent intends to complete but cannot act on immediately. The CDA library models this as `TodoTool`, a persistent checklist backed by `~/.cda/todos.json`. The agent writes `todo(action="add", text="Run tests after refactor")` when it identifies a pending task, and marks it complete with `todo(action="done", id=...)` once finished.

We inspect the `TodoTool` schema via the default registry:

In [ ]:
from notebooks.agent.tools.registry import create_default_registry

registry = create_default_registry(Config())
schemas = registry.get_schemas()
todo_schema = next((s for s in schemas if s["function"]["name"] == "todo"), None)
if todo_schema:
    print(json.dumps(todo_schema, indent=2))

The schema exposes the five actions — `add`, `done`, `delete`, `list`, `clear` — and the two parameters: `text` (required for `add`) and `id` (required for `done` and `delete`). The agent uses this to maintain a running task list across turns, treating prospective memory as a lightweight project manager embedded in the session.

## Memory-Augmented Agent

We now run a live agent session that uses working memory (context window, managed automatically), episodic memory (session persistence), and semantic memory (`MemoryTool`). The agent is given a task that requires storing a fact, and we verify the fact persists after the session is saved and reloaded.

**Setup.** We build an agent restricted to the `memory` tool to keep the demo focused:

In [ ]:
async def run_and_display(agent: Agent, prompt: str) -> str:
    """Stream agent events and print a live trace; return the final text."""
    final = ""
    async for event in agent.run(prompt):
        if event.type == AgentEventType.TEXT_DELTA:
            print(event.data.get("content", ""), end="", flush=True)
        elif event.type == AgentEventType.TOOL_CALL_START:
            tool_name = event.data.get("name", "")
            args = event.data.get("arguments", {})
            print(f"\n[tool: {tool_name}({args})]", end="", flush=True)
        elif event.type == AgentEventType.TOOL_CALL_COMPLETE:
            output = event.data.get("output", "")
            print(f" \u2192 {str(output)[:60]}", end="", flush=True)
        elif event.type == AgentEventType.TEXT_COMPLETE:
            final = event.data.get("content", "")
    print()
    return final

**Experiment.** First turn: the agent stores a database fact to semantic memory.

In [ ]:
config = Config(allowed_tools=["memory"])
agent = Agent(config)

response1 = await run_and_display(
    agent,
    "Please remember that our primary database is PostgreSQL 15 on port 5432.",
)
print(f"\nTurn 1 response: {response1[:200]}")

Saving the session to episodic memory:

In [ ]:
path = agent.session.save("memory-demo")
print(f"Session saved to: {path}")
print(f"Turns: {agent.session.turn_count}, Messages: {len(agent.session.messages)}")

Loading the session in a fresh agent and verifying that semantic memory persisted independently of the session:

In [ ]:
config2 = Config(allowed_tools=["memory"])
loaded_session = Session.load("memory-demo", config2)
print(f"Loaded {len(loaded_session.messages)} messages, {loaded_session.turn_count} turns")

tool = MemoryTool()
result = await tool.execute(ToolInvocation(params={"action": "list"}))
print(f"\nPersisted memory:\n{result.output}")

Cleaning up the demo session and memory entries:

In [ ]:
demo_path = SESSIONS_DIR / "memory-demo.json"
if demo_path.exists():
    demo_path.unlink()
    print("Demo session deleted.")

await tool.execute(ToolInvocation(params={"action": "clear"}))
print("Memory cleared.")

:::{.callout-important}
Memory and session persistence give agents continuity across runs — but they also accumulate stale information. A fact that was true last week may no longer hold: the database may have migrated to a new port, the user's preferred language may have changed, the project structure may have been reorganized. Design memory operations deliberately: use descriptive, namespaced keys (`project.db_port` rather than just `db_port`), prefer `set` over duplicating keys, and periodically audit the store with `list`. Stale semantic memory is harder to debug than an empty one.

:::

---

■